# Step 13: 통합 — 완성된 mini_claude 에이전트

## 학습 목표

- **13개 서브시스템을 하나의 Agent 클래스로 통합**하는 방법을 이해합니다
- **YAML 설정 파일**로 에이전트를 구성하는 방법을 배웁니다
- **전체 파이프라인 데이터 흐름**을 처음부터 끝까지 추적합니다

> **통합의 핵심**
> 지금까지 만든 13개의 서브시스템은 각각 독립적으로 동작합니다.
> 이 Step에서는 이들을 하나의 `Agent` 클래스로 조립하여, "사용자 질문 입력 -> 최종 응답 출력"의 전체 파이프라인을 완성합니다.
> Claude Code의 `src/entrypoints/cli.tsx`가 바로 이 조립 과정을 수행하는 진입점입니다.

---

## Claude Code 소스 분석

### 13-1. assembleToolPool() — 모든 도구 소스 병합 (src/tools.ts:345+)

Claude Code는 여러 출처의 도구를 하나의 풀로 병합합니다:

```typescript
// src/tools.ts:345+ (핵심 간소화)

function assembleToolPool(): Tool[] {
  const tools: Tool[] = []

  // 1. 내장 도구 (Bash, Read, Write, Edit, Grep, Glob...)
  tools.push(...builtinTools)

  // 2. MCP 도구 (외부 서버에서 동적으로 가져온 도구)
  tools.push(...mcpTools)

  // 3. 스킬 도구 (SkillTool — 스킬을 호출하는 도구)
  tools.push(new SkillTool(skillRegistry))

  // 4. 에이전트 도구 (AgentTool — 서브에이전트 생성)
  tools.push(new AgentTool(agentConfig))

  // 5. 커스텀 도구 (사용자가 추가한 도구)
  tools.push(...customTools)

  return tools
}
```

### 13-2. CLI 부트 시퀀스 (src/entrypoints/cli.tsx)

Claude Code의 시작은 다음 순서로 진행됩니다:

```typescript
// src/entrypoints/cli.tsx (부트 시퀀스 간소화)

async function main() {
  // 1. 설정 로드 (CLI 인자 + 환경변수 + 설정 파일)
  const config = loadConfig()

  // 2. 컨텍스트 구성 (시스템 프롬프트 + 프로젝트 정보)
  const context = buildContext(config)

  // 3. 도구 풀 조립
  const tools = assembleToolPool()

  // 4. LLM 클라이언트 초기화
  const client = createClient(config.provider)

  // 5. 에이전트 루프 시작 (REPL 모드)
  await runREPL({ client, context, tools })
}
```

### 13-3. 전체 파이프라인 데이터 흐름

```
Boot → Context Assembly → Compact Check → API Call
  ↓
Hooks (사전 처리) → Permissions → Tool Execution
  ↓
State Update → Recovery (에러 시) → Memory Extraction
  ↓
Next Turn / Done
```

각 단계에서 어떤 서브시스템이 관여하는지:

| 단계 | 서브시스템 | Step |
|------|-----------|------|
| Boot | AgentConfig, LLMClient | 1, 13 |
| Context Assembly | ContextManager, Memory | 5, 9 |
| Compact Check | Compaction | 8 |
| API Call | LLMClient (agent_loop) | 1 |
| Hooks | HookRegistry | 6 |
| Permissions | PermissionChecker | 7 |
| Tool Execution | ToolRegistry, Orchestrator | 2, 3 |
| MCP Tools | MCPClient | 4 |
| Skills | SkillTool, SkillRegistry | 10 |
| Sub-agents | AgentTool | 11 |
| Bridge | BridgeServer | 12 |
| Recovery | RetryConfig, GracefulShutdown | (에이전트 루프 내장) |
| State | State (불변 상태) | 1 |

---

## Python으로 구현하기

### 구현 1: AgentConfig와 YAML 설정 로딩

`mini_claude/agent.py`의 `AgentConfig` 클래스와 `load_config()` 함수를 살펴봅니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.agent import Agent, AgentConfig, load_config
from mini_claude.recovery import RetryConfig

# --- 1. AgentConfig 기본값 확인 ---
config = AgentConfig()
print("=== AgentConfig 기본값 ===")
print(f"  provider:       {config.provider}")
print(f"  model:          {config.model}")
print(f"  max_tokens:     {config.max_tokens}")
print(f"  max_turns:      {config.max_turns}")
print(f"  system_prompt:  {config.system_prompt[:60]}...")
print(f"  bridge_enabled: {config.bridge_enabled}")
print(f"  retry_config:   max_retries={config.retry_config.max_retries}")

# --- 2. YAML 설정 파일 작성 및 로드 ---
import tempfile
from pathlib import Path

config_dir = Path(tempfile.mkdtemp(prefix="config_"))
config_file = config_dir / "agent_config.yaml"

yaml_content = """\
provider: anthropic
model: claude-sonnet-4-20250514
max_tokens: 8192
max_turns: 50
system_prompt: "당신은 mini_claude 워크샵의 AI 어시스턴트입니다."
bridge_enabled: false
"""
config_file.write_text(yaml_content, encoding="utf-8")

print("\n=== YAML 설정 파일에서 로드 ===")
try:
    loaded_config = load_config(config_file)
    print(f"  provider:       {loaded_config.provider}")
    print(f"  model:          {loaded_config.model}")
    print(f"  max_tokens:     {loaded_config.max_tokens}")
    print(f"  max_turns:      {loaded_config.max_turns}")
    print(f"  system_prompt:  {loaded_config.system_prompt}")
except Exception as e:
    print(f"  (yaml 라이브러리 없음 — 기본값 사용: {e})")

# --- 3. kwargs 오버라이드 ---
print("\n=== kwargs로 설정 오버라이드 ===")
print("Agent(provider='anthropic', max_turns=10) 형태로 직접 지정 가능")
print("설정 우선순위: kwargs > YAML 파일 > 기본값")

### 구현 2: Agent 클래스 — 전체 서브시스템 통합

`Agent` 클래스는 모든 서브시스템을 초기화하고, `run()`과 `repl()`로 에이전트를 실행합니다.
Claude Code의 `cli.tsx` 부트 시퀀스를 하나의 클래스로 캡슐화한 것입니다.

In [ ]:
# --- Agent 클래스의 초기화 과정을 단계별로 확인 ---

# Agent 생성 (API 키가 없어도 초기화까지는 가능)
agent = Agent(provider="auto", max_turns=20)

print("=== Agent 초기화 완료 ===\n")

# 1. 설정 확인
print("[1] AgentConfig")
print(f"    provider: {agent.config.provider}")
print(f"    max_turns: {agent.config.max_turns}")

# 2. 도구 레지스트리 확인
print(f"\n[2] ToolRegistry")
tool_names = agent._tool_registry.get_names()
print(f"    등록된 도구 수: {len(tool_names)}")
print(f"    도구 목록: {tool_names}")

# 3. 스킬 레지스트리 확인
print(f"\n[3] SkillRegistry")
skill_names = agent._skill_registry.list_names()
print(f"    등록된 스킬 수: {len(agent._skill_registry)}")
print(f"    스킬 목록: {skill_names}")

# 4. 각 서브시스템의 역할 설명
print(f"\n=== 통합된 서브시스템 ===")
subsystems = [
    ("LLMClient",       "API 호출 추상화 (Anthropic/OpenAI)"),
    ("ToolRegistry",    "도구 등록/조회/스키마 생성"),
    ("Orchestrator",    "도구 실행 오케스트레이션 (직렬/병렬)"),
    ("SkillRegistry",   "스킬 등록/조회"),
    ("SkillTool",       "LLM이 스킬을 호출하는 도구"),
    ("AgentTool",       "서브에이전트 생성 도구"),
    ("RetryConfig",     "에러 복구 (지수 백오프)"),
    ("GracefulShutdown","Ctrl+C 우아한 종료"),
]

for name, desc in subsystems:
    print(f"  {name:20s} — {desc}")

# 5. 전체 파이프라인 (agent.run()의 내부 동작)
print(f"\n=== agent.run() 파이프라인 ===")
pipeline = [
    "1. LLM 클라이언트 지연 초기화 (첫 호출 시)",
    "2. 도구 스키마 생성 (ToolRegistry → Anthropic/OpenAI 형식)",
    "3. 오케스트레이터 생성 (직렬/병렬 자동 판단)",
    "4. retry_with_backoff() 래핑 (에러 시 자동 재시도)",
    "5. run_agent() 호출 → agent_loop() 실행",
    "6.   while True: API 호출 → 응답 처리",
    "7.     도구 호출 있으면 → 오케스트레이터로 실행 → 반복",
    "8.     도구 호출 없으면 → 최종 텍스트 반환",
]
for step in pipeline:
    print(f"  {step}")

---

## 실습 1: 에이전트 실행 — 파일 읽기 + 분석

API 키가 설정되어 있다면, 실제로 에이전트를 실행하여 파일을 읽고 분석하는 전체 파이프라인을 확인합니다.

> **API 키 설정**: `ANTHROPIC_API_KEY` 또는 `OPENAI_API_KEY` 환경변수가 설정되어 있어야 합니다.

In [ ]:
# --- 시나리오: "이 프로젝트의 README를 읽고 요약해줘" ---
# 전체 파이프라인이 동작하는 것을 확인합니다.

# API 키 확인
has_api_key = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"))

if has_api_key:
    agent = Agent(provider="auto", max_turns=10)
    
    print("=== 에이전트 실행: 파일 읽기 + 분석 ===\n")
    result = await agent.run(
        "이 프로젝트의 README.md를 읽고 3줄로 요약해주세요.",
        verbose=True,
    )
    
    print("\n" + "=" * 60)
    print("최종 응답:")
    print(result)
else:
    print("=== API 키가 설정되지 않았습니다 ===\n")
    print("실제 실행을 보려면 다음 환경변수 중 하나를 설정하세요:")
    print("  export ANTHROPIC_API_KEY='sk-...'")
    print("  export OPENAI_API_KEY='sk-...'")
    print()
    print("API 키 없이도 에이전트의 구조와 파이프라인은 위에서 확인할 수 있습니다.")
    print()
    print("--- 예상 실행 흐름 ---")
    print("  [도구] Read({'file_path': 'README.md'})")
    print("  [결과] # mini_claude Workshop...")
    print()
    print("  최종 응답: 이 프로젝트는 Claude Code의 아키텍처를 분석하여...")

# --- REPL 인터페이스 소개 ---
print("\n=== REPL 인터페이스 ===")
print("대화형 세션을 시작하려면:")
print("  agent = Agent()")
print("  await agent.repl()")
print()
print("REPL 내장 명령어:")
print("  /quit   — 종료")
print("  /tools  — 등록된 도구 목록")
print("  /skills — 등록된 스킬 목록")

---

## 실습 2: mini_claude 모듈 전체 구조 확인

지금까지 만든 모든 모듈의 구조를 한눈에 확인합니다.

In [ ]:
# --- mini_claude 패키지 전체 구조 ---
from pathlib import Path

base = Path("mini_claude")
print("=== mini_claude 패키지 구조 ===\n")

# 모듈별 설명
module_descriptions = {
    "__init__.py":          "(패키지 초기화)",
    "llm_client.py":        "Step 1  — LLM API 추상화 (Anthropic + OpenAI)",
    "agent_loop.py":        "Step 1  — 핵심 while(true) 에이전트 루프",
    "tool_base.py":         "Step 2  — Tool 프로토콜/인터페이스",
    "tool_registry.py":     "Step 2  — 도구 등록/조회/스키마 생성",
    "orchestrator.py":      "Step 3  — 직렬/병렬 도구 실행 오케스트레이션",
    "context.py":           "Step 5  — 컨텍스트 관리",
    "hooks.py":             "Step 6  — 훅 시스템 (사전/사후 처리)",
    "permissions.py":       "Step 7  — 도구 실행 권한 관리",
    "compaction.py":        "Step 8  — 컨텍스트 압축",
    "memory.py":            "Step 9  — 메모리 추출/관리",
    "state.py":             "Step 1  — 불변 상태 관리",
    "recovery.py":          "Step 1  — 에러 복구 (재시도, 종료 처리)",
    "agent.py":             "Step 13 — 통합 Agent 클래스",
}

tools_descriptions = {
    "__init__.py":          "(패키지 초기화)",
    "bash_tool.py":         "Step 2  — Bash 명령어 실행 도구",
    "file_read_tool.py":    "Step 2  — 파일 읽기 도구",
    "file_write_tool.py":   "Step 2  — 파일 쓰기 도구",
    "skill_tool.py":        "Step 10 — 스킬 호출 도구",
    "agent_tool.py":        "Step 11 — 서브에이전트 생성 도구",
}

# 메인 모듈
for name, desc in module_descriptions.items():
    filepath = base / name
    exists = filepath.exists()
    status = "OK" if exists else "--"
    print(f"  [{status}] mini_claude/{name:22s} {desc}")

# tools/ 하위 모듈
print(f"\n  --- tools/ ---")
for name, desc in tools_descriptions.items():
    filepath = base / "tools" / name
    exists = filepath.exists()
    status = "OK" if exists else "--"
    print(f"  [{status}] mini_claude/tools/{name:18s} {desc}")

# skills/ 하위 모듈
print(f"\n  --- skills/ ---")
print(f"  [OK] mini_claude/skills/loader.py        Step 10 — 스킬 정의/로딩/레지스트리")

# mcp/ 하위 모듈
print(f"\n  --- mcp/ ---")
print(f"  [OK] mini_claude/mcp/client.py           Step 4  — MCP 클라이언트")

# bridge/ 하위 모듈
print(f"\n  --- bridge/ ---")
print(f"  [OK] mini_claude/bridge/server.py        Step 12 — WebSocket 브리지 서버")

# 총 모듈 수
total = sum(1 for _ in base.rglob("*.py"))
print(f"\n총 Python 모듈 수: {total}개")

---

## 전체 아키텍처 다이어그램

```
                            +---------------------------+
                            |      Agent (Step 13)      |
                            |  - AgentConfig (YAML)     |
                            |  - run() / repl()         |
                            +-------------+-------------+
                                          |
                    +---------------------+---------------------+
                    |                                           |
          +---------+---------+                    +------------+------------+
          |   LLM Client      |                    |     Bridge Server       |
          |   (Step 1)        |                    |     (Step 12)           |
          |   Anthropic/OpenAI|                    |     WebSocket/SSE       |
          +---------+---------+                    +-------------------------+
                    |
          +---------+---------+
          |   Agent Loop      |
          |   (Step 1)        |
          |   while(true) {   |
          |     query()       |
          |     tools?        |
          |   }               |
          +---------+---------+
                    |
    +---------------+---------------+
    |               |               |
+---+---+   +------+------+   +----+----+
|Context|   |  Compaction  |   | Memory  |
|(St.5) |   |   (St.8)    |   | (St.9)  |
+-------+   +-------------+   +---------+

    +---------------+---------------+
    |               |               |
+---+---+   +------+------+   +----+----+
| Hooks |   | Permissions  |   |Recovery |
|(St.6) |   |   (St.7)    |   | (St.1)  |
+-------+   +-------------+   +---------+

    +-----------------------------------------------+
    |            Tool Execution Layer                |
    |  +----------+  +----------+  +----------+     |
    |  |ToolReg.  |  |Orchestr. |  |MCP Client|     |
    |  |(Step 2)  |  |(Step 3)  |  |(Step 4)  |     |
    |  +----------+  +----------+  +----------+     |
    |  +----------+  +----------+                   |
    |  |SkillTool |  |AgentTool |                   |
    |  |(Step 10) |  |(Step 11) |                   |
    |  +----------+  +----------+                   |
    +-----------------------------------------------+
```

---

## 13개 Step에서 배운 핵심 설계 원칙 7가지

### 원칙 1: while(true) + 종료 조건 = 에이전트
에이전트의 핵심은 놀랍도록 단순합니다. LLM을 호출하고, 도구 호출이 있으면 실행하고 반복, 없으면 종료. 모든 복잡성은 이 루프 안의 각 단계에 플러그인으로 끼워 넣어집니다.

### 원칙 2: 불변 상태 + AsyncGenerator = 안전한 스트리밍
매 턴마다 새로운 State 객체를 생성하고, AsyncGenerator로 중간 결과를 yield합니다. 이것이 에러 복구, 디버깅, 실시간 UI를 모두 가능하게 하는 기반입니다.

### 원칙 3: 프로토콜 기반 도구 시스템
모든 도구가 동일한 인터페이스(name, description, input_schema, call)를 따릅니다. 새 도구를 추가할 때 에이전트 루프를 수정할 필요가 없습니다. MCP 외부 도구도 같은 인터페이스로 통합됩니다.

### 원칙 4: 직렬/병렬 자동 오케스트레이션
읽기 전용 도구는 병렬 실행, 쓰기 도구는 직렬 실행. 이 판단을 도구의 `is_read_only()` 메서드로 자동화합니다. 성능과 안전성을 동시에 확보하는 패턴입니다.

### 원칙 5: 선언적 확장 (Hooks, Skills, YAML 설정)
코드를 수정하지 않고도 동작을 변경할 수 있습니다. 훅으로 사전/사후 처리를 추가하고, 스킬로 새 워크플로우를 정의하고, YAML로 설정을 변경합니다.

### 원칙 6: 다중 보호 계층 (Permissions + Hooks + Tool Restrictions)
권한 시스템, 훅, 스킬의 도구 제한이 겹겹이 보호합니다. 하나의 계층이 실패해도 다른 계층이 안전을 보장합니다. 이것이 Claude Code가 사용자의 파일 시스템에서 안전하게 동작하는 비결입니다.

### 원칙 7: 코디네이터는 지시만, 워커가 실행
복잡한 작업은 코디네이터가 계획을 세우고, 전문화된 서브에이전트가 실행합니다. 관심사의 분리(Separation of Concerns)를 에이전트 시스템에 적용한 것입니다.

---

## Claude Code 아키텍처에서 배우는 에이전트 설계 교훈

1. **단순한 코어, 복잡한 플러그인**: 에이전트 루프 자체는 20줄이면 충분합니다. 복잡성은 플러그인(도구, 훅, 스킬)에 넣으세요.

2. **프롬프트가 코드를 대체할 수 있다**: 스킬 시스템이 보여주듯, 잘 작성된 프롬프트는 코드만큼 강력합니다. 새 기능을 추가할 때 "코드로 짤까, 프롬프트로 짤까?"를 먼저 고민하세요.

3. **에러는 반드시 발생한다**: API 타임아웃, 토큰 초과, 권한 거부. Claude Code는 각 에러에 대한 자동 복구 전략을 미리 준비합니다. 에이전트 개발에서 "해피 패스"만 구현하면 실전에서 실패합니다.

4. **캐시를 활용한 비용 최적화**: 시스템 프롬프트 상속으로 프롬프트 캐시를 공유하고, 컨텍스트 압축으로 토큰을 절약합니다. 에이전트 비용의 대부분은 API 호출입니다.

5. **관찰 가능성(Observability)**: AsyncGenerator의 이벤트 스트림, 상태 전이 기록, 도구 실행 로그. 에이전트가 "왜 이렇게 동작했는지"를 추적할 수 있어야 디버깅이 가능합니다.

---

이것으로 **mini_claude 워크샵**을 마칩니다. 13개의 Step을 통해 Claude Code의 핵심 아키텍처를 분석하고, Python으로 직접 구현해보았습니다. 이 경험이 여러분만의 에이전트 시스템을 설계하고 구축하는 데 도움이 되기를 바랍니다.